# 02 — Feature Matrix
Daily features shared by entry filters and ML models: VIX rank, RSI, realized
vol, vol-risk premium, trend. Everything is computed from close-of-day data —
no lookahead.

In [ ]:
import sys
sys.path.insert(0, "../src")
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.width", 160)

In [ ]:
from lab.market_data import load_market
from lab.features import build_features, ML_FEATURES

feats = build_features(load_market())
feats.tail()

In [ ]:
ax = feats.set_index("quote_date")[["vix", "realized_vol_20d"]].plot(title="Implied (VIX) vs realized vol")
ax.set_ylabel("%");

In [ ]:
feats.set_index("quote_date")[["vix_rank", "rsi14"]].plot(subplots=True, title=["VIX 1y percentile rank", "SPX RSI(14)"]);

## Regime scatter: where do high-vol / oversold days live?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(feats.rsi14, feats.vix_rank, c=feats.quote_date.dt.year, s=4, cmap="viridis")
ax.set_xlabel("RSI(14)"); ax.set_ylabel("VIX rank"); plt.colorbar(sc, label="year");

In [ ]:
# How often is each candidate entry filter true?
for expr in ["vix_rank > 0.5", "vix_rank > 0.8", "rsi14 < 30", "vix_rank > 0.5 and rsi14 < 40"]:
    print(f"{expr:40s} {feats.eval(expr).mean():6.1%} of days")